# ドラマ info scrapping

Two sources:
- http://dorama.info/drama/drama_season.php <br>
with parameter ?year=2021&season=4&ut=0&fd=0&md=0&dk=1 , says.
- https://zh.wikipedia.org/wiki/日本劇集列表 <br>
with parameter \_(2020年) , says.

## Get html from dorama.info

In [1]:
import urllib.request, urllib.parse, urllib.error
from datetime import date
from Scarper import Scarper

# import importlib
# importlib.reload(Scarper)

year = 2021
season = 3
verbose = True

service_url = 'http://dorama.info/drama/drama_season.php'
params = {
    'year':year
    , 'season':season
    , 'ut':0
    , 'fd':0
    , 'md':0
    , 'dk':1
}

url = service_url + '?' + urllib.parse.urlencode(params)

scarper = Scarper()
html = scarper.crawl(url, verbose)

Retrieving http://dorama.info/drama/drama_season.php?year=2021&season=3&ut=0&fd=0&md=0&dk=1
Retrieved 202872 characters


Extract the table called "table_ss" in the page

In [2]:
from bs4 import BeautifulSoup 
soup = BeautifulSoup(html, 'html.parser')

In [3]:
soup_table = soup.find_all("table", attrs={"class": "table_ss"})
soup_rows = soup_table[0].find_all('tr', recursive=False)
print(f'{len(soup_rows)} dramas have been detected.')

93 dramas have been detected.


Row 0: Column names <br>
Row 1+: each drama's demo

In [4]:
data_rows = []
column_names = ['TV', 'BC_time', 'BC_date', 'name', '_', 'rate', 'watch_rating', 'recommend_ppl', 'popularity', 'taiwan', 'actors']

for soup_row_raw in soup_rows:
    soup_row = soup_row_raw.find_all('td', recursive=False, attrs={'class': 'td_ss line_bot'})
    if len(soup_row)==0:    continue
    # soup_row[0] is drama name
    # soup_row[1] is drama preformance
    data_row = [word.strip() for tag in soup_row[0].find_all('td', attrs={'class': 'td2_g'}) for word in tag.text.replace('\n', '').split('\xa0')] + \
        [tag.text.replace('\n', '').strip() for tag in soup_row[1].find_all('td', attrs={'class': 'td2_g'})]
    # DQ-change: if the 0th element too long, then splited by space.
    if len(data_row[0])>10:
        data_row_0_new = data_row.pop(0).split()
        data_row = data_row_0_new + data_row
    
    if len(data_row) != len(column_names):
        if verbose:
            print(f'column_names: {column_names}')
            print(f'data_row: {data_row}')
            print(f'length not match. SKIP.')
        continue
    dict_row = dict(zip(column_names, data_row))
    dict_row['year'] = year
    dict_row['season'] = season
    dict_row['BC_date'] = date(year, int(dict_row['BC_date'][:2]), int(dict_row['BC_date'][-2:]))
    dict_row['BC_weekday'] = dict_row['BC_date'].isoweekday()
    data_rows.append(dict_row)

column_names: ['TV', 'BC_time', 'BC_date', 'name', '_', 'rate', 'watch_rating', 'recommend_ppl', 'popularity', 'taiwan', 'actors']
data_row: ['BS', 'TV東', '24', '07-10', '我想被表揚的妄想飯', '', '', '', '9', '1592', '', '高杉真宙']
length not match. SKIP.
column_names: ['TV', 'BC_time', 'BC_date', 'name', '_', 'rate', 'watch_rating', 'recommend_ppl', 'popularity', 'taiwan', 'actors']
data_row: ['0', '08-30', '拉麵刑警', '', '', '', '2', '376', '', '津田寬治,篠田諒']
length not match. SKIP.
column_names: ['TV', 'BC_time', 'BC_date', 'name', '_', 'rate', 'watch_rating', 'recommend_ppl', 'popularity', 'taiwan', 'actors']
data_row: ['23', '08-28', '拉麵刑警', '', '', '', '2', '376', '', '津田寬治,篠田諒']
length not match. SKIP.
column_names: ['TV', 'BC_time', 'BC_date', 'name', '_', 'rate', 'watch_rating', 'recommend_ppl', 'popularity', 'taiwan', 'actors']
data_row: ['21', '09-26', '拉麵刑警', '', '', '', '2', '376', '', '津田寬治,篠田諒']
length not match. SKIP.
column_names: ['TV', 'BC_time', 'BC_date', 'name', '_', 'rate', 'watch

In [5]:
import pandas as pd
pd.DataFrame(data_rows)

,TV,BC_time,BC_date,name,_,rate,watch_rating,recommend_ppl,popularity,taiwan,actors,year,season,BC_weekday
0,CX,21,2021-06-21,夜間醫生,,4.4 / 5 8 人,10.98目前,87,20943,已播,"波瑠,田中圭",2021,3,1
1,TX,23,2021-05-31,●名偵探主廚,,4.4 / 5 20 人,2.62,176,12737,播過,西島秀俊,2021,3,1
2,CX,21,2021-07-06,她很漂亮 (6天結),,2.8 / 5 5 人,6.81目前,60,15607,已播,"中島健人,小芝風花",2021,3,2
3,EX,21,2021-07-07,刑事7人7 (7天結),,5 / 5 1 人,11.29目前,24,4692,,"東山紀之,田邊誠一",2021,3,3
4,EX,20,2021-07-01,IP~網路搜查班 (8天結),,4 / 5 1 人,8.87目前,110,10704,,"佐佐木藏之介,福原遙",2021,3,4
5,EX,21,2021-07-08,女王偵訊室4 (8天結),,4.2 / 5 6 人,11.80目前,198,11389,已播,"天海祐希,田中哲司",2021,3,4
6,TX,20,2021-04-30,●警視廳0系第五季,,4.7 / 5 6 人,,176,4223,,"小泉孝太郎,松下由樹",2021,3,5
7,TBS,22,2021-07-09,#家族募集中,,4.3 / 5 4 人,6.57目前,54,9199,已播,"重岡大毅,木村文乃",2021,3,5
8,NTV,22,2021-07-10,VOICEII,,,9.10目前,158,8474,已播,"唐澤壽明,真木陽子",2021,3,6
9,TX,23,2021-07-03,女之戰爭,,4 / 5 1 人,,13,2470,,"古川雄大,葵若菜",2021,3,6


## Get html from wikipedia

In [1]:
import urllib.request, urllib.parse, urllib.error
from datetime import date
from Scarper import Scarper

year = 2021
verbose = True
service_url = 'https://zh.wikipedia.org/wiki/'
url = service_url + urllib.parse.quote(f'日本劇集列表_({year}年)')

scarper = Scarper()
html = scarper.crawl(url, verbose)

Retrieving https://zh.wikipedia.org/wiki/%E6%97%A5%E6%9C%AC%E5%8A%87%E9%9B%86%E5%88%97%E8%A1%A8_%282021%E5%B9%B4%29
Retrieved 297344 characters


Extract all tables

In [2]:
from bs4 import BeautifulSoup 
soup = BeautifulSoup(html, 'html.parser')

Search the table title tag, with class name = "mw-headline"

In [3]:
import re
soup_tables = soup.find_all(id=re.compile(str(year)), attrs={'class': 'mw-headline'})

then for each title, find the next table.

In [4]:
soup_rows = soup_tables[0].find_next('table').find_all('tr')

Row 0: Column names <br>
Row 1: 2nd row of column names <br>
Row 2+: Each drama's demo

In [5]:
def soup_row_preprocessor(soup_row_td, year, season, service_url):
    
    dict_row = {}
    
    dict_row['date'] = get_text_br_spliter(str(soup_row_td[0]))
    dict_row['time'] = get_text_br_spliter(str(soup_row_td[1]))
    dict_row['weekday'] = soup_row_td[2].get_text(strip=True)

    str_temp = get_text_br_spliter(str(soup_row_td[3]))
    dict_row['name_zh'] = str_temp[0]
    dict_row['name_ja'] = str_temp[0]
    if len(str_temp)==2:    dict_row['name_ja'] = str_temp[1]
    url_wikis = [urllib.parse.urljoin(service_url, tag['href']) for tag in soup_row_td[3].find_all(href=True)]
    dict_row['url_wiki_zh'] = next((u for u in url_wikis if u.find('zh.wikipedia.org') > 0), None)
    dict_row['url_wiki_ja'] = get_url_wiki_ja(dict_row['name_ja'])

    dict_row['TV'] = soup_row_td[5].get_text(strip=True)
    dict_row['char_main'] = soup_row_td[6].get_text(strip=True)
    dict_row['char_others'] = soup_row_td[7].get_text(strip=True).replace('等', '').split('、')
    
    dict_row['url'] = None
    if soup_row_td[8].find('a'):    dict_row['url'] = soup_row_td[8].a['href']
    dict_row['theme_song'] = soup_row_td[9].get_text(strip=True)
    
    dict_row['year'] = year
    dict_row['season'] = season
    
    if dict_row['url_wiki_ja'] is not None:
        dict_row_individual_wiki = crawl_infobox(dict_row['url_wiki_ja'])
        dict_row.update(dict_row_individual_wiki)
    
    print('soup_row_preprocessor is completed.', dict_row['name_zh'], dict_row['name_ja'])
#     print(dict_row)
    
    return dict_row

In [11]:
dict_rows = []

for soup_table in soup_tables[0:4]:
    
    season = soup_tables.index(soup_table) + 1
    soup_rows = soup_table.find_next('table').find_all('tr')
    for soup_row in soup_rows[2:]:
        soup_row_td = soup_row.find_all('td', recursive=False)

        dict_row = soup_row_preprocessor(soup_row_td, year, season, service_url)  
        dict_rows.append(dict_row)


soup_row_preprocessor is completed.
小女侍
おちょやん
soup_row_preprocessor is completed.
監察醫 朝顏(第二季）
監察医 朝顔（第2シリーズ）
soup_row_preprocessor is completed.
匿名者～警視廳“指殺人” 對策室（日语：アノニマス〜警視庁“指殺人”対策室〜）
アノニマス〜警視庁“指殺人”対策室〜
soup_row_preprocessor is completed.
在澡堂等著你（日语：でっけぇ風呂場で待ってます）
でっけぇ風呂場で待ってます
soup_row_preprocessor is completed.
青之SP ～學校內警察．嶋田隆平～（日语：青のSP―学校内警察・嶋田隆平―）
青のSP―学校内警察・嶋田隆平―
soup_row_preprocessor is completed.
Oh! My Boss!戀愛在別冊
オー!マイ・ボス!恋は別冊で
soup_row_preprocessor is completed.
我家女兒交不到男朋友！！
ウチの娘は、彼氏が出来ない!!
soup_row_preprocessor is completed.
激辛道
ゲキカラドウ
soup_row_preprocessor is completed.
大叔與貓
おじさまと猫
soup_row_preprocessor is completed.
テレビ演劇 サクセス荘3（日语：テレビ演劇 サクセス荘3）
テレビ演劇 サクセス荘3
soup_row_preprocessor is completed.
認識的妻子
知ってるワイフ
soup_row_preprocessor is completed.
虹色病歷簿
にじいろカルテ
soup_row_preprocessor is completed.
遺留搜查(第六季)
遺留捜査（第6シーズン）
soup_row_preprocessor is completed.
江戶小姐（日语：江戸モアゼル〜令和で恋、いたしんす。〜）
江戸モアゼル〜令和で恋、いたしんす。〜
soup_row_preprocessor is completed.
你不是犯人（日语：あなた犯人じゃありません）
あなた犯人じゃありません
soup_

In [13]:
import pandas as pd
pd.set_option("max_columns", 100)

df = pd.DataFrame(dict_rows)

In [15]:
df.head(1)

,date,time,weekday,name_zh,name_ja,url_wiki_zh,url_wiki_ja,TV,char_main,char_others,url,theme_song,year,season,ジャンル,作,演出,出演者,ナレーター,音楽,オープニング,国・地域,言語,時代設定,制作統括,プロデューサー,製作,放送チャンネル,映像形式,音声形式,放送国・地域,放送期間,放送時間,放送枠,放送分,回数,脚本,監督,監修,エンディング,チーフ・,制作,配信サイト,配信国・地域,配信期間,企画,原案,別名,話数,製作総指揮,ディレクター,プロデュース,原作,シーズン数,シリーズ数,各話の長さ,声の出演,著者,発行日,発行元,国,形態,ページ数,公式サイト,コード,撮影地,前作,配信時間,イラスト,役職,ポジション(現役時),背番号(現役時),身長(現役時),体重(現役時),本名,愛称,ラテン文字,誕生日,出身地,出身,ドラフト,1983-19911991-19921992-19941994-1996,1999-20032004-20132013-20202020-,シンボル,親,兄弟,子供,放送局
0,"[上一季, |03月26日]","[08時00分, |08時15分]",一至五,小女侍,おちょやん,https://zh.wikipedia.org/wiki/%E5%B0%8F%E5%A5%...,https://ja.wikipedia.org/wiki/%E3%81%8A%E3%81%...,NHK,杉咲花,"[松本敦, 宮澤艾瑪, 篠原涼子, 名倉潤, 石野陽子, 成田凌, 星田英利, 井川遙, 若...",https://www.nhk.or.jp/dramatopics-blog/1000/41...,主：秦基博「泣き笑いのエピソード」,2021,1,[テレビドラマ],[八津弘幸],"[梛川善郎, 盆子原誠]","[杉咲花, 成田凌, 名倉潤, 宮田圭子, いしのようこ, 宮澤エマ, 星田英利, 板尾創路...",[桂吉弥（語り・黒衣役）],[サキタハヂメ],[秦基博「泣き笑いのエピソード」],[日本],[日本語],[大正-昭和],"[櫻井壮一, 熊野律時]",[村山峻平],[NHK大阪放送局],[NHK総合],[文字多重放送],[解説放送],[日本],[2020年11月30日-2021年5月14日],[月曜 - 金曜 8:00 - 8:15],[連続テレビ小説],[15分],[125[1]→ 未定[2]→115[3]],NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Get URL for individual drama from wikipedia

In [71]:
import urllib.request, urllib.parse, urllib.error
from datetime import date
from Scarper import Scarper

import json

name_ja = 'もこみちのMIDNIGHT KITCHEN ドラマ'

service_url = 'https://ja.wikipedia.org/w/api.php'
params = {
    'action': 'query'
    , 'list': 'search'
    , 'srsearch': name_ja
    , 'format': 'json'
    , 'srlimit': 10
}

url = service_url + '?' + urllib.parse.urlencode(params)
verbose = True

scarper = Scarper()
js = scarper.crawl(url, verbose)

dict_js = json.loads(js)

Retrieving https://ja.wikipedia.org/w/api.php?action=query&list=search&srsearch=%E3%82%82%E3%81%93%E3%81%BF%E3%81%A1%E3%81%AEMIDNIGHT+KITCHEN+%E3%83%89%E3%83%A9%E3%83%9E&format=json&srlimit=10
Retrieved 10157 characters


In [72]:
dict_js["query"]["search"]

[{'ns': 0,
  'title': '速水もこみち',
  'pageid': 175911,
  'size': 31855,
  'wordcount': 3799,
  'snippet': '役 緊急取調室 FORTH SEASON（2021年7月8日 - 9月16日） <span class="searchmatch">もこみちの</span><span class="searchmatch">MIDNIGHT</span> <span class="searchmatch">KITCHEN</span>（2014年2月27日 - 3月20日、日本テレビ） - 主演・本人(主要登場人物) 役 奇跡<span class="searchmatch">の</span>教室（2014年6月28日、日本テレビ） - 西川久 役 地獄先生ぬ〜べ〜（2014年10月11日',
  'timestamp': '2021-09-19T17:33:40Z'},
 {'ns': 0,
  'title': 'もこみちのMIDNIGHT KITCHEN',
  'pageid': 2941348,
  'size': 4320,
  'wordcount': 318,
  'snippet': '『<span class="searchmatch">もこみちの</span><span class="searchmatch">MIDNIGHT</span> <span class="searchmatch">KITCHEN</span>』（<span class="searchmatch">もこみちの</span>ミッドナイトキッチン）は、2014年2月27日から3月20日まで日本テレビで放送していたテレビ<span class="searchmatch">ドラマ</span>。 日本テレビ朝番組『ZIP!』<span class="searchmatch">の</span>「MOCO\'Sキッチン」と連動しているフェイクドキュメンタリー<span class="searchmatch">ドラマ</span>。番組放送後1週間公式HPで未公開部分を含む動画を公開した。',
  'timestamp': '2019-07-07T05:41:29Z'},
 {'ns': 0,

In [51]:
out_url = ''
if dict_js["query"]["search"]:    
    out_url = 'https://ja.wikipedia.org/wiki/' + urllib.parse.quote(dict_js["query"]["search"][0]["title"])
print(out_url)

Testing

In [16]:
def get_url_wiki_ja(name_ja, verbose=False):
    
    import urllib.request, urllib.parse, urllib.error
    from datetime import date
    from Scarper import Scarper

    import json

    service_url = 'https://ja.wikipedia.org/w/api.php'
    params = {
        'action': 'query'
        , 'list': 'search'
        , 'srsearch': name_ja
        , 'format': 'json'
        , 'srlimit': 1
    }

    url = service_url + '?' + urllib.parse.urlencode(params)

    scarper = Scarper()
    js = scarper.crawl(url, verbose)

    dict_js = json.loads(js)
    
    out_url = ''
    if dict_js["query"]["search"]:    
        out_url = 'https://ja.wikipedia.org/wiki/' + urllib.parse.quote(dict_js["query"]["search"][0]["title"])
    return out_url

get_url_wiki_ja('ネメシス')

'https://ja.wikipedia.org/wiki/%E3%83%8D%E3%83%A1%E3%82%B7%E3%82%B9'

## Get html from individual drama from wikipedia

In [51]:
import urllib.request, urllib.parse, urllib.error
from datetime import date
from Scarper import Scarper

url = 'https://ja.wikipedia.org/wiki/%E6%B3%A3%E3%81%8F%E3%81%AA%E7%A0%94%E4%BF%AE%E5%8C%BB'
verbose = True

scarper = Scarper()
html = scarper.crawl(url, verbose)

Retrieving https://ja.wikipedia.org/wiki/%E6%B3%A3%E3%81%8F%E3%81%AA%E7%A0%94%E4%BF%AE%E5%8C%BB
Retrieved 90769 characters


In [52]:
from bs4 import BeautifulSoup 
soup = BeautifulSoup(html, 'html.parser')

Get Infobox (the big box in right hand side of wiki)

In [64]:
infoboxes = soup.find_all('table', attrs={'class':'infobox'})

In [65]:
infobox_rows = []
for infobox in infoboxes:
    if 'テレビ番組' in infobox.find_all('tr', attrs={'class': 'noprint'})[0].a['title']:
        infobox_rows = infobox.find_all('tr')
        break

In [66]:
def get_text_br_spliter(html):
    from bs4 import BeautifulSoup, NavigableString, Tag
    
    soup = BeautifulSoup(html, 'html.parser')
    text = ''
    for child in soup.find_all()[0]:
        if isinstance(child, NavigableString):
            text += str(child).strip()
        elif isinstance(child, Tag):
            if child.name != 'br':
                text += child.text.strip()
            else:
                text += '\n'

    result = text.strip().split('\n')
    return result

In [67]:
dict_rows = {}
for infobox_row in infobox_rows:
    row_keys = infobox_row.find_all('th', attrs={'scope':'row'})
    if not row_keys:    continue
    row_key_bs = row_keys[0]
    row_key = get_text_br_spliter(str(row_key_bs))[0]

    row_value_bs = row_key_bs.find_next('td') 
    row_value = get_text_br_spliter(str(row_value_bs))
    dict_rows[row_key] = row_value

In [68]:
dict_rows

{'ジャンル': ['連続ドラマ'],
 '原作': ['中山祐次郎『泣くな研修医』'],
 '脚本': ['樋口卓治'],
 '演出': ['豊島圭介', '朝鳥ツワ子', '小松隆志'],
 '監修': ['中澤暁雄（医療）', '山本昌督（医療）', '根本千草（医療）'],
 '出演者': ['白濱亜嵐', '木南晴夏', '野村周平', '柄本時生', '恒松祐里', '山口智充', '木村昴'],
 '音楽': ['☆Taku Takahashi（m-flo）'],
 'エンディング': ['DOBERMAN INFINITY', '「konomama」'],
 '国・地域': ['日本'],
 '言語': ['日本語'],
 '製作総指揮': ['横地郁英（テレビ朝日）'],
 'プロデューサー': ['服部宣之（テレビ朝日）', '残間理央（テレビ朝日）', '山本喜彦（MMJ）', '小路美智子（MMJ）'],
 '製作': ['テレビ朝日', 'メディアミックス・ジャパン'],
 '放送チャンネル': ['テレビ朝日系'],
 '映像形式': ['文字多重放送'],
 '音声形式': ['ステレオ放送'],
 '放送国・地域': ['日本'],
 '放送期間': ['2021年4月24日-6月26日'],
 '放送時間': ['土曜 23:00 - 23:30'],
 '放送枠': ['土曜ナイトドラマ'],
 '放送分': ['30分'],
 '回数': ['10']}

Testing

In [70]:
def crawl_infobox(url, verbose=False):

    import urllib.request, urllib.parse, urllib.error
    from datetime import date
    from Scarper import Scarper    
    
    if not url:    return {}
    
    scarper = Scarper()
    html = scarper.crawl(url, verbose)  

    from bs4 import BeautifulSoup 
    soup = BeautifulSoup(html, 'html.parser')
    
    infoboxes = soup.find_all('table', attrs={'class':'infobox'})
    if not infoboxes:    return {}    
    
    infobox_rows = []
    for infobox in infoboxes:
        if 'テレビ番組' in infobox.find_all('tr', attrs={'class': 'noprint'})[0].a['title']:
            infobox_rows = infobox.find_all('tr')
            break    
    
    dict_rows = {}
    for infobox_row in infobox_rows:
        row_keys = infobox_row.find_all('th', attrs={'scope':'row'})
        if not row_keys:    continue
        row_key_bs = row_keys[0]
        row_key = get_text_br_spliter(str(row_key_bs))[0]

        row_value_bs = row_key_bs.find_next('td') 
        row_value = get_text_br_spliter(str(row_value_bs))
        dict_rows[row_key] = row_value
        
    return dict_rows

crawl_infobox('https://ja.wikipedia.org/wiki/%E3%82%82%E3%81%93%E3%81%BF%E3%81%A1%E3%81%AEMIDNIGHT_KITCHEN')

{'ジャンル': ['テレビドラマ'],
 '企画': ['三枝孝臣'],
 '脚本': ['塩沢航'],
 '演出': ['長尾くみこ、CUプロダクション演出グループ'],
 '出演者': ['速水もこみち'],
 'プロデューサー': ['三枝孝臣'],
 '制作': ['日本テレビ'],
 '音声形式': ['ステレオ放送'],
 '放送国・地域': ['日本'],
 '放送期間': ['2014年2月27日 - 3月20日'],
 '放送時間': ['木曜 25:24 - 25:39'],
 '放送分': ['15分'],
 '回数': ['4']}